# Your first PINN with PINNeAPPle: heat in a plate, with its physics checks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PINNeAPPle-Labs/PINNeAPPle/blob/main/notebooks/colab/01_first_pinn_heat.ipynb)

Steady heat conduction in a square plate is Laplace's equation, $\nabla^2 T = 0$ on $[0,1]^2$. The top edge is held
at $T = \sin(\pi x)$, the other three edges at $T = 0$. The exact solution is
$T = \sin(\pi x)\,\sinh(\pi y)/\sinh(\pi)$, so every number below can be checked.

What this notebook does:

1. describes the problem once (geometry, equation, boundary conditions);
2. solves it with a physics-informed network (`pp.solve(..., "pinn")`) and with finite elements (`"fem"`) through the same call;
3. checks the result the way any result should be checked before it is used: the PDE residual, the boundary
   conditions, conservation of energy, the maximum principle and the error against a reference, summarised in a
   **trust card** with a decision (APPROVED, REVIEW or REJECT).

Runs top to bottom on a free Colab GPU (Runtime > Change runtime type > T4 GPU) in about a minute, or on a CPU in about
three minutes.

In [ ]:
# Setup: on Colab (or wherever PINNeAPPle is missing) install it from PyPI, then pick the device.
import importlib.util, os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB or importlib.util.find_spec("pinneapple") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pinneapple>=0.7.0"], check=True)

import numpy as np
import matplotlib.pyplot as plt
import torch
import pinneapple as pp

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PINNeAPPle {pp.__version__} on {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else " (no GPU: slower, still fine)"))

In [ ]:
# Parameters. SMOKE=True shrinks every size so the notebook runs in seconds (the CI runs it that way).
SMOKE = os.environ.get("PINNEAPPLE_NB_SMOKE", "0") == "1"
EPOCHS = 150 if SMOKE else 2000        # PINN training epochs
N_COLLOCATION = 512 if SMOKE else 2048  # interior points where the PDE is enforced
HIDDEN = 32 if SMOKE else 48           # width of the network
FEM_N = 16 if SMOKE else 48            # finite-element cells per axis
GRID = 33 if SMOKE else 65             # points per axis of the checking grid

## 1. The problem

`ProblemBuilder` writes the problem as data: the domain, the field, the equation and each boundary condition with
the edge it applies to. `pp.PhysicalProblem` is the one description every solver in the library accepts.

In [ ]:
from pinneapple_physics.pde_environment.builder import ProblemBuilder

def top_edge(X):
    return np.sin(np.pi * X[:, 0])

spec = (ProblemBuilder("heated_plate").domain(x=(0, 1), y=(0, 1)).fields("T").pde("laplace")
        .bc("dirichlet", field="T", value=top_edge, on="y_max")
        .bc("dirichlet", field="T", value=0.0, on="y_min")
        .bc("dirichlet", field="T", value=0.0, on="x_boundary")
        .build())
problem = pp.PhysicalProblem.from_pde_spec(spec)

def exact(X):
    return (np.sin(np.pi * X[:, 0]) * np.sinh(np.pi * X[:, 1]) / np.sinh(np.pi))[:, None]

print(problem.summary())

## 2. Solve it twice: a PINN and finite elements

`pp.solve(problem, method, **options)` trains or runs the method and returns a `Solution` with `predict(X)` and the
same `metadata()` for every backend. No training loop to write.

In [ ]:
pinn = pp.solve(problem, "pinn", epochs=EPOCHS, n_collocation=N_COLLOCATION, hidden_dim=HIDDEN, n_layers=4,
                device=DEVICE, seed=0)
fem = pp.solve(problem, "fem", n=FEM_N)
for s in (pinn, fem):
    m = s.metadata()
    print(f"{m['method']:>4} ({m['kind']}): {m['wall_time_s']:.1f} s")

In [ ]:
# Both against the exact solution, at the same points
print(pp.compare(problem, [pinn, fem], reference="exact", options={"exact": {"fn": exact}}))

In [ ]:
x = y = np.linspace(0, 1, GRID)
X, Y = np.meshgrid(x, y, indexing="ij")
P = np.stack([X.ravel(), Y.ravel()], axis=1)
T_pinn = pinn.predict(P)[:, 0].reshape(X.shape)
T_fem = fem.predict(P)[:, 0].reshape(X.shape)
T_exact = exact(P)[:, 0].reshape(X.shape)

fig, axs = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, (title, v, cmap) in zip(axs, [("PINN", T_pinn, "inferno"), ("FEM", T_fem, "inferno"),
                                      ("|PINN - exact|", np.abs(T_pinn - T_exact), "viridis")]):
    im = ax.pcolormesh(x, y, v.T, shading="auto", cmap=cmap)
    fig.colorbar(im, ax=ax)
    ax.set(title=title, xlabel="x", ylabel="y", aspect="equal")
plt.tight_layout()
plt.show()

## 3. Check the physics: the trust card

A small error against one reference is not enough to trust a field. The trust card checks, on a grid:

| check | what it asks |
|---|---|
| PDE residual | does $\nabla^2 T$ vanish inside the plate (finite differences)? |
| boundary conditions | is $T$ equal to the prescribed value on each edge? |
| conservation | does the heat entering through the top leave through the other edges (divergence theorem)? |
| maximum principle | is no interior temperature hotter or colder than the edges? |
| reference | relative L2 error against the exact solution |
| applicability (OOD) | are the queried points inside the region the model was trained on? |

A failing **critical** check (residual, a boundary condition, the reference) rejects the result; any other failure
asks for review. `pp.trust_card` is new after 0.7.0: on an older install the cell loads the same module from the
repository.

In [ ]:
from pathlib import Path
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/PINNeAPPle-Labs/PINNeAPPle/main/"

def repo_file(rel):
    """A file of the PINNeAPPle repository: from the local checkout if there is one, else downloaded from GitHub."""
    roots = [Path(os.environ["PINNEAPPLE_REPO"])] if os.environ.get("PINNEAPPLE_REPO") else []
    for root in roots + [Path.cwd(), *Path.cwd().parents]:
        if (root / rel).is_file() and (root / "pinneapple").is_dir():
            return root / rel
    dest = Path("pinneapple_files") / rel
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

def load_trust_card():
    try:
        return pp.trust_card
    except AttributeError:                  # PINNeAPPle 0.7.0: load the module from the repository
        spec = importlib.util.spec_from_file_location("field_card",
                                                      repo_file("pinneapple_analysis/trust/field_card.py"))
        mod = importlib.util.module_from_spec(spec)
        sys.modules["field_card"] = mod
        spec.loader.exec_module(mod)
        return mod.trust_card

trust_card = load_trust_card()

In [ ]:
edges = {"x_min": 0.0, "x_max": 0.0, "y_min": 0.0, "y_max": top_edge}
card = trust_card({"x": x, "y": y}, T_pinn, name="PINN", pde="laplace", dirichlet=edges, reference=T_exact,
                  train_bounds={"x": (0, 1), "y": (0, 1)})
print(card)

In [ ]:
# a grid solution is checked on its own nodes: finite differences across the kinks of an interpolated
# piecewise-linear field would measure the interpolation, not the solution
fx, fy = fem.grid["x"], fem.grid["y"]
FX, FY = np.meshgrid(fx, fy, indexing="ij")
FP = np.stack([FX.ravel(), FY.ravel()], axis=1)
card_fem = trust_card({"x": fx, "y": fy}, fem.predict(FP)[:, 0].reshape(FX.shape), name="FEM", is_ml=False,
                      pde="laplace", dirichlet=edges, reference=exact(FP)[:, 0].reshape(FX.shape))
print(card_fem)

The card also answers the six questions a reviewer asks of any physics result (data and geometry, model, physical
constraints, benchmark, uncertainty, engineering decision), each from the checks above:

In [ ]:
for stage in card.stages.values():
    print(f"{stage['title']:<22} {stage['status']:<9} {stage.get('next', '')}")

## What to try next

* `03_equal_budget_benchmark.ipynb`: compare PINN variants and FEM with the same training budget.
* `05_verify_a_model.ipynb`: the trust card catching bad predictions that still look plausible.
* Change the boundary condition or add a source (`.pde("poisson")` with `ctx={"source_fn": ...}`) and re-run the
  checks: the card tells you when the network has not learned the physics.